# Silver Cleaning

Applies every Silver transform from `src/silver/transforms.py` to the Bronze parquet and measures before versus after. Then demonstrates Type 2 versioning with `src/silver/scd2.py` and scores the result with the quality gates.

## Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
BRONZE = ROOT / "data" / "bronze"
SILVER = ROOT / "data" / "silver"

from src.quality.gates import check_not_null, check_unique, score
from src.silver.scd2 import CUSTOMER_TRACKED, apply_scd2, asof_join
from src.silver.transforms import (
    clean_customers,
    clean_orders,
    clean_products,
    drop_extract_meta,
    enrich_returns,
    union_orders,
)

print("imports ok")

imports ok


## Customers

Dedupe, case fix, phone type, and date parsing.

In [2]:
raw_cust = pd.read_parquet(BRONZE / "dim_customers.parquet")
cust = clean_customers(raw_cust)
print(
    f"rows {len(raw_cust)} -> {len(cust)}, dupes removed: {raw_cust.duplicated(subset=['customer_id']).sum()}"
)
print("gender values:", sorted(cust["gender"].dropna().unique()))
print("null customer_id:", int(cust["customer_id"].isna().sum()))
cust.head(3)

rows 3050 -> 3000, dupes removed: 50
gender values: ['Female', 'Male']
null customer_id: 0


,customer_id,full_name,gender,birth_date,phone,email,city,region,loyalty_tier,registration_date,_batch_id
0,CUS01518,Mohamed Mansour,Male,1981-05-01,1158132110,mohamed.mansour52@hotmail.com,Suez,Canal Zone,Bronze,2020-12-19,41717df1-895c-4d2b-8f0c-90c007b0fd3c
1,CUS02370,Maged Lotfy,Male,2003-04-10,1281872177,NaN,Zagazig,Nile Delta,Bronze,2021-05-13,41717df1-895c-4d2b-8f0c-90c007b0fd3c
2,CUS01962,Ahmed Hafez,Male,1971-05-03,1141856715,ahmed.hafez18@outlook.com,Giza,Greater Cairo,Gold,2023-09-26,41717df1-895c-4d2b-8f0c-90c007b0fd3c


## Products

Pipe split into name, color, and size plus redundant column drop.

In [3]:
raw_prod = pd.read_parquet(BRONZE / "dim_products.parquet")
prod = clean_products(raw_prod)
print(f"rows {len(raw_prod)} -> {len(prod)}")
prod[["product_id", "product_name", "color", "size", "unit_price"]].head(5)

rows 345 -> 345


,product_id,product_name,color,size,unit_price
0,CLT0001,T-Shirt,Black,S,90
1,CLT0002,T-Shirt,Black,M,90
2,CLT0003,T-Shirt,Black,L,90
3,CLT0004,T-Shirt,Black,XL,90
4,CLT0005,T-Shirt,White,S,90


## Orders

Trim, date parse, key filter, then append of both frames.

In [4]:
first = clean_orders(pd.read_parquet(BRONZE / "fact_orders_2022_2023.parquet"))
second = clean_orders(pd.read_parquet(BRONZE / "fact_orders_2024.parquet"))
orders = union_orders(first, second)
print(f"frames {len(first)} + {len(second)} -> union {len(orders)}")
print("date range:", orders["order_date"].min(), "to", orders["order_date"].max())
print("dup order_id:", int(orders.duplicated(subset=["order_id"]).sum()))

frames 7942 + 4058 -> union 12000
date range: 2022-01-01 to 2024-12-31
dup order_id: 0


## Returns

Enrichment with item counts and revenue from order details.

In [5]:
rets = enrich_returns(
    drop_extract_meta(pd.read_parquet(BRONZE / "fact_returns.parquet")),
    drop_extract_meta(pd.read_parquet(BRONZE / "fact_order_details.parquet")),
)
print(f"rows: {len(rets)}, orphans flagged: {int(rets['return_orphan'].sum())}")
rets[["return_id", "order_id", "n_items", "order_revenue", "return_orphan"]].head(3)

rows: 1056, orphans flagged: 0


,return_id,order_id,n_items,order_revenue,return_orphan
0,RET00001,ORD0000010,1,85.0,False
1,RET00002,ORD0000017,2,8285.0,False
2,RET00003,ORD0000029,1,180.0,False


## SCD Type 2 demo

Version a customer sample, then simulate a region move and show the old version closing plus a new current version opening.

In [6]:
sample = cust.head(200).copy()
v1 = apply_scd2(
    None, sample, "customer_id", CUSTOMER_TRACKED, "customer_sk", "2022-01-01"
)
moved = sample.copy()
moved.loc[moved.index[:5], "region"] = "Moved Region"
v2 = apply_scd2(v1, moved, "customer_id", CUSTOMER_TRACKED, "customer_sk", "2024-01-01")
print(f"rows {len(v1)} -> {len(v2)}, closed versions: {int((~v2['is_current']).sum())}")
v2[v2["customer_id"].isin(moved["customer_id"].iloc[:5])][
    ["customer_id", "region", "customer_sk", "is_current"]
]

rows 200 -> 205, closed versions: 5

,customer_id,region,customer_sk,is_current
0,CUS01518,Canal Zone,1,False
1,CUS02370,Nile Delta,2,False
2,CUS01962,Greater Cairo,3,False
3,CUS00344,Greater Cairo,4,False
4,CUS02664,Nile Delta,5,False
200,CUS01518,Moved Region,201,True
201,CUS02370,Moved Region,202,True
202,CUS01962,Moved Region,203,True
203,CUS00344,Moved Region,204,True
204,CUS02664,Moved Region,205,True


## Point in time join

Facts resolve to the dimension version active on the order date.

In [7]:
joined = asof_join(orders.head(2000), v2, "customer_id", "customer_sk", "order_date")
print(f"facts in: 2000, joined rows: {len(joined)}")
assert len(joined) == 2000

facts in: 2000, joined rows: 2000

## Quality gates

Null, uniqueness, and referential checks scored as a suite.

In [8]:
results = {}
results.update(
    {f"cust_null_{k}": v for k, v in check_not_null(cust, ["customer_id"]).items()}
)
results.update(
    {f"cust_unique_{k}": v for k, v in check_unique(cust, ["customer_id"]).items()}
)
results["orders_unique_order_id"] = int(orders.duplicated(subset=["order_id"]).sum())
results["returns_orphan"] = int(rets["return_orphan"].sum())
pct, failed = score(results)
print(f"suite score: {pct:.1f} percent, failed checks: {failed}")
pd.DataFrame([{"check": k, "failures": v} for k, v in results.items()])

suite score: 100.0 percent, failed checks: 0


,check,failures
0,cust_null_customer_id,0
1,cust_unique_customer_id,0
2,orders_unique_order_id,0
3,returns_orphan,0


## Summary

Silver is deduped, typed, trimmed, appended, enriched, and versioned. The next notebook builds the Gold star schema and marts on top of these outputs.